# Figure 2 — Subtype determination and demographics

- **SubA**: Elbow plot (total within-cluster sum of squares vs. number of clusters) used to choose K=2 for K-means subtyping.
- **SubB**: Age distribution and sex composition across TDC, Subtype L, and Subtype H.
- **SubC**: Cortical maps of the mean Z-score for each subtype (Subtype L: volume reductions; Subtype H: volume increases).

## Panel A — Clustering validity indices (ABIDE-II and CABIC)

In [ ]:
"""Panel A: internal clustering validity indices (silhouette, Calinski-Harabasz,
Davies-Bouldin) for K-means on the ASD Z-score profiles of both cohorts."""
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# --- Global style configuration ---
GLOBAL_CONFIG = {'DPI': 300, 'FIGURE_WIDTH': 6.2, 'FIGURE_HEIGHT': 3.0,
                 'FONT_SIZE_MAIN': 16, 'FONT_SIZE_LABEL': 12,
                 'FONT_SIZE_TICK': 11, 'FONT_SIZE_LEGEND': 10}
plt.rcParams.update({'font.size': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'axes.labelsize': GLOBAL_CONFIG['FONT_SIZE_LABEL'],
                     'xtick.labelsize': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'ytick.labelsize': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'legend.fontsize': GLOBAL_CONFIG['FONT_SIZE_LEGEND'],
                     'font.family': 'serif', 'font.serif': ['Times New Roman'],
                     'axes.unicode_minus': False, 'savefig.dpi': GLOBAL_CONFIG['DPI'],
                     'figure.dpi': GLOBAL_CONFIG['DPI']})

# --- Paths and parameters ---
PATHS = {
    'ABIDE2': 'output/ABIDE2_Morpho_Zscore_aparc.csv',
    'CABIC': 'output/CABIC_Morpho_Zscore_aparc.csv',
}
SAVE_DIR = 'Fig2'
SAVE_PATH = os.path.join(SAVE_DIR, 'SubA.png')
os.makedirs(SAVE_DIR, exist_ok=True)
K_MIN, K_MAX = 2, 6


def get_all_cluster_metrics(file_path, dataset_name):
    """Compute the three clustering indices for K in [K_MIN, K_MAX]."""
    print(f"[{dataset_name}] processing...")
    if not os.path.exists(file_path):
        print(f"Error: file not found {file_path}")
        return None
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Read CSV failed: {e}")
        return None
    feature_cols = [col for col in df.columns if 'GrayVol' in col]
    if not feature_cols:
        print(f"Warning: no 'GrayVol' columns in {dataset_name}")
        return None
    X = df[feature_cols].values
    print(f"  samples: {X.shape[0]}, features: {X.shape[1]}")
    metrics = {'Silhouette': {}, 'CH_Index': {}, 'DB_Index': {}}
    for k in range(K_MIN, K_MAX + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = kmeans.fit_predict(X)
        metrics['Silhouette'][k] = silhouette_score(X, labels)
        metrics['CH_Index'][k] = calinski_harabasz_score(X, labels)
        metrics['DB_Index'][k] = davies_bouldin_score(X, labels)
    print(f"  done (K={K_MIN}-{K_MAX})")
    return metrics


print("=== Computing clustering indices ===")
all_results = {}
for name, path in PATHS.items():
    result = get_all_cluster_metrics(path, name)
    if result is not None:
        all_results[name] = result
if not all_results:
    print("No data could be processed; aborting.")
    sys.exit(1)

# --- Plotting ---
fig, axs = plt.subplots(1, 3, figsize=(GLOBAL_CONFIG['FIGURE_WIDTH'], GLOBAL_CONFIG['FIGURE_HEIGHT']))
fig.suptitle('Clustering Internal Validity Indices Comparison',
             fontweight='bold', fontsize=GLOBAL_CONFIG['FONT_SIZE_MAIN'])
styles = {'ABIDE2': {'color': '#8DA0CB', 'marker': 'o', 'label': 'ABIDE-II'},
          'CABIC': {'color': '#66C2A5', 'marker': 's', 'label': 'CABIC'}}
metric_keys = ['Silhouette', 'CH_Index', 'DB_Index']
ylabs = ['Avg Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin']

for i, m_key in enumerate(metric_keys):
    ax = axs[i]
    for ds_name, result in all_results.items():
        ks = list(result[m_key].keys())
        vals = list(result[m_key].values())
        ax.plot(ks, vals, linestyle='-', linewidth=2, markersize=9, **styles[ds_name])
        best_k = min(result[m_key], key=result[m_key].get) if m_key == 'DB_Index' else max(result[m_key], key=result[m_key].get)
        ax.scatter(best_k, result[m_key][best_k], color='red', s=40, edgecolors='black', zorder=5, linewidth=0.8)
    ax.set_xlabel('Clusters (K)')
    ax.set_ylabel(ylabs[i])
    ax.set_xticks(range(K_MIN, K_MAX + 1))
    ax.grid(True, linestyle='--', alpha=0.4)
    if i == 1:
        ax.legend(loc='best', frameon=True)
    sns.despine(ax=ax)

plt.savefig(SAVE_PATH, dpi=GLOBAL_CONFIG['DPI'], bbox_inches='tight')
print(f"\nImage saved: {SAVE_PATH}")
plt.show()

## Panel B — Demographic characteristics

In [ ]:
"""Panel B: demographic matching between Subtype L and Subtype H
(age violin plots and sex stacked bar plots) for both cohorts."""
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# --- Global style configuration (1x4 layout) ---
GLOBAL_CONFIG = {'DPI': 300, 'FIGURE_WIDTH': 6.2, 'FIGURE_HEIGHT': 3.0,
                 'FONT_SIZE_MAIN': 16, 'FONT_SIZE_LABEL': 11,
                 'FONT_SIZE_TICK': 10, 'FONT_SIZE_LEGEND': 9}
plt.rcParams.update({'font.size': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'axes.titlesize': GLOBAL_CONFIG['FONT_SIZE_MAIN'],
                     'axes.labelsize': GLOBAL_CONFIG['FONT_SIZE_LABEL'],
                     'xtick.labelsize': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'ytick.labelsize': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'legend.fontsize': GLOBAL_CONFIG['FONT_SIZE_LEGEND'],
                     'font.family': 'serif', 'font.serif': ['Times New Roman'],
                     'savefig.dpi': GLOBAL_CONFIG['DPI'], 'figure.dpi': GLOBAL_CONFIG['DPI'],
                     'axes.unicode_minus': False})

COLORS_SUBTYPE = {'Subtype L': '#8DA0CB', 'Subtype H': '#66C2A5'}
COLORS_SEX = {'F': '#FC8D62', 'M': '#89CFF0'}
PATHS = {'ABIDE-II': 'data/ABIDE2_AGE_Sub.xlsx',
         'CABIC': 'data/CABIC_AGE_Sub.xlsx'}
SAVE_PATH = 'Fig2/SubB.png'


def run_demographic_analysis():
    fig, axes = plt.subplots(1, 4, figsize=(GLOBAL_CONFIG['FIGURE_WIDTH'],
                                            GLOBAL_CONFIG['FIGURE_HEIGHT']))
    fig.suptitle('Demographic Matching: Age and Sex Distribution',
                 fontweight='bold', fontsize=GLOBAL_CONFIG['FONT_SIZE_MAIN'])
    for i, (name, path) in enumerate(PATHS.items()):
        if not os.path.exists(path):
            print(f"Skip: file not found {path}")
            continue
        try:
            df = pd.read_excel(path)
        except Exception as e:
            print(f"Read failed {name}: {e}")
            continue
        df = df[df['SUBTYPE_LABEL'].isin([0, 1])].copy()
        df['SUBTYPE_LABEL'] = df['SUBTYPE_LABEL'].map({0: 'Subtype L', 1: 'Subtype H'})
        n_l = (df['SUBTYPE_LABEL'] == 'Subtype L').sum()
        n_h = (df['SUBTYPE_LABEL'] == 'Subtype H').sum()
        print(f"[{name}] Subtype L = {n_l}, Subtype H = {n_h}, total = {len(df)}")
        gL = df[df['SUBTYPE_LABEL'] == 'Subtype L']
        gH = df[df['SUBTYPE_LABEL'] == 'Subtype H']
        _, p_age = stats.ttest_ind(gL['AGE'].dropna(), gH['AGE'].dropna())
        contingency = pd.crosstab(df['SUBTYPE_LABEL'], df['SEX'])
        _, p_sex, _, _ = stats.chi2_contingency(contingency)
        print(f"[{name}] Age P: {p_age:.4f} | Sex P: {p_sex:.4f}")

        idx_age, idx_sex = i * 2, i * 2 + 1
        # Age violin plot
        ax_age = axes[idx_age]
        sns.violinplot(ax=ax_age, x='SUBTYPE_LABEL', y='AGE', data=df,
                       palette=COLORS_SUBTYPE, inner='quartile', linewidth=1.0,
                       order=['Subtype L', 'Subtype H'])
        ax_age.set_title(f'{name}\nAge', pad=5, fontsize=12)
        ax_age.set_xlabel('')
        ax_age.set_ylabel('Age (years)')
        ax_age.set_xticklabels(['L', 'H'])
        sns.despine(ax=ax_age)
        # Sex stacked bar plot
        ax_sex = axes[idx_sex]
        sex_pct = pd.crosstab(df['SUBTYPE_LABEL'], df['SEX'], normalize='index') * 100
        sex_pct = sex_pct.reindex(['Subtype L', 'Subtype H'])
        plot_sex_cols = [c for c in ['F', 'M'] if c in sex_pct.columns]
        plot_colors = [COLORS_SEX[c] for c in plot_sex_cols]
        sex_pct[plot_sex_cols].plot(kind='bar', stacked=True, ax=ax_sex, color=plot_colors,
                                    width=0.7, edgecolor='white', linewidth=0.8, rot=0)
        ax_sex.set_title(f'{name}\nSex', pad=5, fontsize=12)
        ax_sex.set_xlabel('')
        ax_sex.set_ylabel('Percentage (%)')
        ax_sex.set_xticklabels(['L', 'H'])
        if idx_sex == 3:
            ax_sex.legend(title='Sex', loc='upper left', frameon=True, fontsize=8,
                          bbox_to_anchor=(1.05, 1.0))
        elif ax_sex.get_legend():
            ax_sex.get_legend().remove()
        for p in ax_sex.patches:
            h = p.get_height()
            if h > 15:
                ax_sex.text(p.get_x() + p.get_width()/2, p.get_y() + h/2, f'{h:.0f}%',
                            ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        sns.despine(ax=ax_sex)
    plt.savefig(SAVE_PATH, dpi=GLOBAL_CONFIG['DPI'], bbox_inches='tight')
    print(f"\nSaved to: {SAVE_PATH}")
    plt.show()


if __name__ == "__main__":
    run_demographic_analysis()

## Panel C — Z-score deviation maps across 68 cortical regions

In [ ]:
"""Panel C: cortical maps of the mean Z-score deviation for each subtype
(Subtype L / Subtype H) in both cohorts, with a shared colorbar."""
import os
import subprocess
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colorbar as mcolorbar
from enigmatoolbox.utils.parcellation import parcel_to_surface
from enigmatoolbox.plotting import plot_cortical

# Optional headless display
xvfb_process = None
try:
    display_num = 99
    xvfb_process = subprocess.Popen(
        ['Xvfb', f':{display_num}', '-screen', '0', '1920x1080x24'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(0.5)
    os.environ['DISPLAY'] = f':{display_num}'
    print(f"[Xvfb] started (:{display_num})")
except Exception as e:
    print(f"[Xvfb] could not start: {e}")

# --- Global style configuration ---
GLOBAL_CONFIG = {'DPI': 300, 'FIGURE_WIDTH': 16, 'FIGURE_HEIGHT': 6,
                 'FONT_SIZE_MAIN': 24, 'FONT_SIZE_LABEL': 28, 'FONT_SIZE_TICK': 22}
SAVE_DIR = 'Fig2'
os.makedirs(SAVE_DIR, exist_ok=True)
plt.rcParams.update({'font.size': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'axes.titlesize': GLOBAL_CONFIG['FONT_SIZE_MAIN'],
                     'font.family': 'serif', 'font.serif': ['Times New Roman'],
                     'savefig.dpi': GLOBAL_CONFIG['DPI']})

# --- Data configs and colormap ---
data_configs = [
    {"name": "ABIDE2", "zscore_file": "output/ABIDE2_Morpho_Zscore_aparc.csv",
     "subtype_file": "data/ABIDE2_AGE_Sub.xlsx"},
    {"name": "CABIC", "zscore_file": "output/CABIC_Morpho_Zscore_aparc.csv",
     "subtype_file": "data/CABIC_AGE_Sub.xlsx"},
]
subtype_mapping = {0: "L", 1: "H"}
custom_cmap_name = 'fig2_rdbu_style'
colors_list = ['#053061', '#2166ac', '#4393c3', '#92c5de', '#f7f7f7',
               '#f4a582', '#d6604d', '#b2182b', '#67001f']
continuous_cmap = LinearSegmentedColormap.from_list(custom_cmap_name, colors_list)
try:
    plt.colormaps.register(cmap=continuous_cmap, name=custom_cmap_name, force=True)
except Exception:
    pass

dk68_labels = [
    'lh_bankssts', 'lh_caudalanteriorcingulate', 'lh_caudalmiddlefrontal', 'lh_cuneus',
    'lh_entorhinal', 'lh_fusiform', 'lh_inferiorparietal', 'lh_inferiortemporal',
    'lh_isthmuscingulate', 'lh_lateraloccipital', 'lh_lateralorbitofrontal', 'lh_lingual',
    'lh_medialorbitofrontal', 'lh_middletemporal', 'lh_parahippocampal', 'lh_paracentral',
    'lh_parsopercularis', 'lh_parsorbitalis', 'lh_parstriangularis', 'lh_pericalcarine',
    'lh_postcentral', 'lh_posteriorcingulate', 'lh_precentral', 'lh_precuneus',
    'lh_rostralanteriorcingulate', 'lh_rostralmiddlefrontal', 'lh_superiorfrontal',
    'lh_superiorparietal', 'lh_superiortemporal', 'lh_supramarginal', 'lh_frontalpole',
    'lh_temporalpole', 'lh_transversetemporal', 'lh_insula',
    'rh_bankssts', 'rh_caudalanteriorcingulate', 'rh_caudalmiddlefrontal', 'rh_cuneus',
    'rh_entorhinal', 'rh_fusiform', 'rh_inferiorparietal', 'rh_inferiortemporal',
    'rh_isthmuscingulate', 'rh_lateraloccipital', 'rh_lateralorbitofrontal', 'rh_lingual',
    'rh_medialorbitofrontal', 'rh_middletemporal', 'rh_parahippocampal', 'rh_paracentral',
    'rh_parsopercularis', 'rh_parsorbitalis', 'rh_parstriangularis', 'rh_pericalcarine',
    'rh_postcentral', 'rh_posteriorcingulate', 'rh_precentral', 'rh_precuneus',
    'rh_rostralanteriorcingulate', 'rh_rostralmiddlefrontal', 'rh_superiorfrontal',
    'rh_superiorparietal', 'rh_superiortemporal', 'rh_supramarginal', 'rh_frontalpole',
    'rh_temporalpole', 'rh_transversetemporal', 'rh_insula'
]


def generate_brain_plot(final_values, output_path, vmin, vmax):
    """Render a 68-vector onto the fsa5 surface."""
    values_fsa5 = parcel_to_surface(final_values, 'aparc_fsa5')
    plot_cortical(array_name=values_fsa5, surface_name="fsa5", size=(1200, 300),
                  cmap=custom_cmap_name, color_bar=False, color_range=(vmin, vmax),
                  screenshot=True, filename=output_path, background=(1, 1, 1), scale=(3, 3))


# --- First pass: compute mean Z per subtype and the global color range ---
print("=== First pass: computing global color range ===")
all_values = {}
global_max = 0.0
for config in data_configs:
    print(f"\n{'='*20} Processing: {config['name']} {'='*20}")
    df_zscore = pd.read_csv(config['zscore_file'])
    df_subtype = pd.read_excel(config['subtype_file'])
    df_combined = pd.merge(df_zscore, df_subtype[['SUBID', 'SUBTYPE_LABEL']], on='SUBID', how='inner')
    print(f"  merged samples: {len(df_combined)}")
    for numeric_id, label_name in subtype_mapping.items():
        df_sub = df_combined[df_combined['SUBTYPE_LABEL'] == numeric_id]
        if df_sub.empty:
            continue
        print(f"  Subtype {label_name}: {len(df_sub)}")
        values_68 = np.zeros(68)
        for i, label in enumerate(dk68_labels):
            col_name = f"Zscore_{label}_GrayVol"
            if col_name in df_sub.columns:
                values_68[i] = df_sub[col_name].mean()
        all_values[(config['name'], label_name)] = values_68
        cur_max = max(abs(values_68.min()), abs(values_68.max()))
        if cur_max > global_max:
            global_max = cur_max
global_max = 0.5 if global_max == 0 else global_max
print(f"\n>>> Global color range: +/-{global_max:.4f}")

# --- Second pass: save 4 individual brain plots (with L/R labels) ---
print("\n=== Second pass: saving individual brain plots ===")
brain_dir = os.path.join(SAVE_DIR, 'SubC_BrainPlots')
os.makedirs(brain_dir, exist_ok=True)
saved_files = []
for (name, label_name), values_68 in all_values.items():
    raw_path = os.path.join(brain_dir, f"raw_{name}_Subtype{label_name}.png")
    generate_brain_plot(values_68, raw_path, vmin=-global_max, vmax=global_max)
    img = plt.imread(raw_path)
    fig, ax = plt.subplots(figsize=(GLOBAL_CONFIG['FIGURE_WIDTH'] * 0.45,
                                    GLOBAL_CONFIG['FIGURE_HEIGHT'] * 0.45))
    ax.imshow(img)
    ax.axis('off')
    h, w, _ = img.shape
    ax.text(w * 0.03, h * 0.1, 'L', fontsize=GLOBAL_CONFIG['FONT_SIZE_LABEL'] * 0.45,
            fontweight='bold', color='black')
    ax.text(w * 0.94, h * 0.1, 'R', fontsize=GLOBAL_CONFIG['FONT_SIZE_LABEL'] * 0.45,
            fontweight='bold', color='black')
    final_path = os.path.join(SAVE_DIR, f"SubC_{name}_Subtype{label_name}_MeanZ.png")
    plt.savefig(final_path, bbox_inches='tight', dpi=GLOBAL_CONFIG['DPI'])
    plt.close(fig)
    saved_files.append(final_path)
    print(f"  [OK] {final_path}")
    os.remove(raw_path)

# --- Third pass: save a standalone colorbar ---
print("\n=== Third pass: saving standalone colorbar ===")
cb_fig, cb_ax = plt.subplots(figsize=(0.4, 3.0))
cb_ax.axis('off')
cax = cb_fig.add_axes([0.3, 0.1, 0.3, 0.8])
norm = plt.Normalize(vmin=-global_max, vmax=global_max)
cb = mcolorbar.ColorbarBase(cax, cmap=continuous_cmap, norm=norm, orientation='vertical')
ticks = [-global_max, 0, global_max]
cb.set_ticks(ticks)
cb.set_ticklabels([f"{t:.2f}" for t in ticks])
cb.ax.tick_params(labelsize=GLOBAL_CONFIG['FONT_SIZE_TICK'])
for label in cb.ax.get_yticklabels():
    label.set_family('serif')
colorbar_path = os.path.join(SAVE_DIR, 'SubC_Colorbar.png')
cb_fig.savefig(colorbar_path, bbox_inches='tight', dpi=GLOBAL_CONFIG['DPI'], transparent=True)
plt.close(cb_fig)
print(f"  [OK] Colorbar saved to: {colorbar_path}")

shutil.rmtree(brain_dir, ignore_errors=True)
if xvfb_process is not None:
    xvfb_process.terminate()
    xvfb_process.wait()
    print("[Xvfb] closed")
print(f"\n=== All files saved to {SAVE_DIR}/ ===")